In [1]:
import pandas as pd
import plotly.express as px

In [2]:
base_rze = {
    "candidate_name": "RZE_BASE",
    "i1": 0.6,
    "i2": 6.0,
    "i3": 11.3,
    "i4": 2.7,
    "o1": 0.3,
    "o2": 3.5,
    "candidate_efficiency": None,
    "candidate_efficient": None,
}

In [10]:
# Ścieżka do CSV z Javy
CSV_PATH = "output/boundary_resampled_results.csv"

# WYBIERASZ TYLKO TE KOLUMNY KTÓRE CHCESZ WIDZIEĆ
PLOT_COLUMNS = ["i1", "i4", "o2"]  # <- zmieniasz tylko to

# Etykiety osi (opcjonalne)
AXIS_LABELS = {
    "i1": "Input 1",
    "i2": "Input 2",
    "i3": "Input 3",
    "i4": "Input 4",
    "o1": "Output 1",
    "o2": "Output 2",
}

# Kolorowanie
COLOR_BY = "candidate_efficient"  
# alternatywa:
# COLOR_BY = "candidate_efficiency"

# Filtrowanie
SHOW_ONLY = "all"  
# "all" / "efficient" / "inefficient"

# Rozmiar punktów
POINT_SIZE = 5

In [11]:
def validate_columns(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def filter_df(df, mode):
    if mode == "all":
        return df
    elif mode == "efficient":
        return df[df["candidate_efficient"] == True].copy()
    elif mode == "inefficient":
        return df[df["candidate_efficient"] == False].copy()
    else:
        raise ValueError("SHOW_ONLY must be: all / efficient / inefficient")


def prepare_bool_column(df):
    if df["candidate_efficient"].dtype == object:
        df["candidate_efficient"] = (
            df["candidate_efficient"]
            .astype(str)
            .str.lower()
            .map({"true": True, "false": False})
        )
    return df

In [12]:
df = pd.read_csv(CSV_PATH)

df = prepare_bool_column(df)

validate_columns(df, PLOT_COLUMNS + ["candidate_name", "candidate_efficient"])

df.head()

,candidate_name,i1,i2,i3,i4,o1,o2,candidate_efficiency,candidate_efficient,SZZ_efficiency,LCJ_efficiency,KAT_efficiency
0,RZE_resampled_0,0.32000,6.0,11.3,1.400000,0.3,2.500000,1.0,True,1.0,0.896000,1.0
1,RZE_resampled_1,0.35000,6.0,11.3,1.400000,0.3,2.500000,1.0,True,1.0,0.980000,1.0
2,RZE_resampled_2,0.38000,6.0,11.3,1.400000,0.3,2.500000,1.0,True,1.0,1.000000,1.0
3,RZE_resampled_3,0.28314,6.0,11.3,1.397834,0.3,2.522659,1.0,True,1.0,0.812587,1.0
4,RZE_resampled_4,0.32000,6.0,11.3,1.380000,0.3,2.500000,1.0,True,1.0,0.896000,1.0


In [13]:
df_plot = filter_df(df, SHOW_ONLY)

print("Number of points:", len(df_plot))

Number of points: 9885


In [14]:
import plotly.express as px
import plotly.graph_objects as go

X_COL, Y_COL, Z_COL = PLOT_COLUMNS

hover_cols = [
    c for c in [
        "candidate_name",
        "candidate_efficiency",
        "candidate_efficient",
        "i1", "i2", "i3", "i4",
        "o1", "o2"
    ]
    if c in df_plot.columns
]

fig = px.scatter_3d(
    df_plot,
    x=X_COL,
    y=Y_COL,
    z=Z_COL,
    color=COLOR_BY,
    hover_name="candidate_name",
    hover_data=hover_cols,
    title=f"DEA search space: {X_COL}, {Y_COL}, {Z_COL}"
)

fig.update_traces(marker=dict(size=POINT_SIZE))

# bazowy punkt RZE
fig.add_trace(
    go.Scatter3d(
        x=[base_rze[X_COL]],
        y=[base_rze[Y_COL]],
        z=[base_rze[Z_COL]],
        mode="markers+text",
        name="RZE base",
        text=["RZE base"],
        textposition="top center",
        marker=dict(
            size=10,
            color="red",
            symbol="diamond"
        ),
        hovertemplate=(
            f"<b>{base_rze['candidate_name']}</b><br>"
            f"{X_COL}: {base_rze[X_COL]}<br>"
            f"{Y_COL}: {base_rze[Y_COL]}<br>"
            f"{Z_COL}: {base_rze[Z_COL]}<extra></extra>"
        )
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title=AXIS_LABELS.get(X_COL, X_COL),
        yaxis_title=AXIS_LABELS.get(Y_COL, Y_COL),
        zaxis_title=AXIS_LABELS.get(Z_COL, Z_COL),
    )
)

fig

In [15]:
fig.write_html("dea_3d_plot_new.html")

In [17]:
import pandas as pd
import plotly.graph_objects as go

# ===== CONFIG =====
file_path = "output/final.csv"   # <-- zmień jeśli trzeba

# ===== LOAD =====
df = pd.read_csv(file_path)

# split klas
df_true = df[df["candidate_efficient"] == True]
df_false = df[df["candidate_efficient"] == False]

# ===== PLOT =====
fig = go.Figure()

# FALSE (np. czerwone)
fig.add_trace(
    go.Scatter3d(
        x=df_false["i1"],
        y=df_false["i4"],
        z=df_false["o2"],
        mode="markers",
        name="efficient_false",
        marker=dict(
            size=5,
            symbol="circle"
        ),
        text=df_false.get("candidate_name", None),
        hovertemplate=(
            "FALSE<br>"
            "candidate: %{text}<br>"
            "i1: %{x}<br>"
            "i4: %{y}<br>"
            "o2: %{z}<extra></extra>"
        )
    )
)

# TRUE (np. niebieskie)
fig.add_trace(
    go.Scatter3d(
        x=df_true["i1"],
        y=df_true["i4"],
        z=df_true["o2"],
        mode="markers",
        name="efficient_true",
        marker=dict(
            size=5,
            symbol="diamond"
        ),
        text=df_true.get("candidate_name", None),
        hovertemplate=(
            "TRUE<br>"
            "candidate: %{text}<br>"
            "i1: %{x}<br>"
            "i4: %{y}<br>"
            "o2: %{z}<extra></extra>"
        )
    )
)

fig.update_layout(
    title="Boundary (k-NN) in 3D space",
    scene=dict(
        xaxis_title="i1 (input ↓)",
        yaxis_title="i4 (input ↓)",
        zaxis_title="o2 (output ↑)"
    ),
    width=1000,
    height=800,
)

fig.show()

In [18]:
fig.write_html("dea_3d_plot_resampled.html")